In [1]:
# Importing libraries
import pandas as pd
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns; sns.set()
from sklearn.decomposition import PCA
from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler 
import warnings
warnings.filterwarnings('ignore')
import scipy
from sklearn.cluster import KMeans
from PIL import Image
import statsmodels.api
from itertools import combinations

In [2]:
# Importing coordinates
local = pd.read_csv(r"C:\Users\joaoa\Desktop\Doutoramento\Professora\Nova pasta\Todos os Pontos (temperatura)\local.cvs")

# Defining parameters
min_corr = 0.6 # 0.6
Y = range(1950, 2023) # 1950-2023
K = range(2, 7) # 2-7
T = ["no restriction", "9h-9h"]
H = [1,2,3,4,5,6]

In [3]:
max_precipitations = pd.read_csv(r"D:\Precipitation\by_water_year\max_precipitations_(mm).csv")
max_precipitations

,year,type_of_period,duration,P0000,P0001,P0002,P0003,P0004,P0005,P0006,...,P1047,P1048,P1055,P1056,P1057,P1058,P1059,P1062,P1063,P1064
0,1950_1951,no restriction,1h,8.326463,8.243036,8.805152,9.028085,9.084161,8.476909,7.380027,...,3.436994,3.357664,3.744721,3.943034,3.918417,3.777545,3.583334,4.295899,4.206998,4.134510
1,1950_1951,no restriction,2h,15.106078,16.101751,16.759608,16.780123,16.432732,15.006237,13.470327,...,6.239377,5.898822,7.132474,7.441573,7.321216,7.192655,7.053152,8.147297,7.940779,7.769816
2,1950_1951,no restriction,3h,20.434577,22.103149,22.261806,21.800894,22.388998,21.266129,19.032700,...,7.886067,8.359287,10.383461,10.725383,10.626908,10.405345,10.334227,11.299809,11.242369,11.195865
3,1950_1951,no restriction,4h,23.979615,25.218735,26.281431,26.591893,26.449654,24.742782,22.127772,...,8.817460,9.966318,14.006458,14.470104,13.779424,13.340397,13.321247,14.879042,14.202038,13.961323
4,1950_1951,no restriction,5h,26.202101,28.151052,28.884135,28.930634,28.555889,26.844913,24.644312,...,9.346757,11.343576,16.998954,17.623985,17.085116,16.196122,15.906174,18.063011,17.503629,17.063234
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3353,2022_2023,9h-9h,24h,54.779407,84.152669,78.163899,69.087140,61.068974,52.875238,47.976034,...,33.152206,27.432676,23.512300,24.586378,25.095019,25.851524,27.461352,29.078925,29.064724,29.050523
3354,2022_2023,9h-9h,48h,87.853795,110.887036,107.861828,103.527183,96.785700,88.935407,80.281787,...,38.806304,32.095753,46.217747,48.270375,49.124993,49.509700,50.263621,50.270073,50.356567,50.426278
3355,2022_2023,9h-9h,72h,94.963124,117.144329,117.125774,116.913173,110.899793,101.122091,88.095980,...,38.814284,32.113306,47.071073,49.186960,50.157759,50.774843,51.829558,51.357064,51.497780,51.613968
3356,2022_2023,9h-9h,96h,114.451494,142.719587,142.209170,141.910078,135.120828,124.048291,109.234199,...,45.058457,43.492522,67.695491,71.006805,73.210482,74.905513,76.910375,76.315241,77.277005,78.081276


In [4]:
# Collecting all information in one dictionary
dict_max = {"no restriction" : {}, "9h-9h" : {}}

for type_of_period in T: 
    for hours in H:
        dict_max[type_of_period][str(hours) + "h"] = max_precipitations[(max_precipitations["type_of_period"] == type_of_period) & (max_precipitations["duration"] == str(hours) + "h")].iloc[:, 3:].reset_index(drop = True)

In [5]:
t = "no restriction"
h = 3
df = dict_max[t][str(h) + "h"]
df

,P0000,P0001,P0002,P0003,P0004,P0005,P0006,P0007,P0008,P0009,...,P1047,P1048,P1055,P1056,P1057,P1058,P1059,P1062,P1063,P1064
0,20.434577,22.103149,22.261806,21.800894,22.388998,21.266129,19.032700,19.820487,21.245612,21.179965,...,7.886067,8.359287,10.383461,10.725383,10.626908,10.405345,10.334227,11.299809,11.242369,11.195865
1,16.214354,17.877661,17.995240,17.503418,17.136343,16.720515,16.341973,17.686952,18.426839,18.261943,...,12.021597,12.265762,12.177706,11.984237,11.197029,10.795709,10.891780,11.223007,11.300437,11.364963
2,12.288716,16.644776,15.489314,14.519313,14.670081,14.443262,13.974937,17.586753,17.474679,17.592093,...,11.182424,10.561522,11.528600,11.937611,12.112547,12.745768,13.749808,12.333065,12.363862,12.388501
3,15.142718,18.652976,17.752420,16.245745,16.309502,16.090598,15.335333,19.523965,20.022906,18.487893,...,24.338432,25.115792,13.793156,14.048191,13.540246,12.848463,13.030173,13.378723,12.885654,12.472287
4,16.324380,18.468801,18.748278,18.750403,18.646263,17.588930,16.044904,17.842902,18.768471,19.026691,...,10.392684,10.806721,14.406305,16.296752,16.443394,16.260622,16.398765,13.809094,13.325591,13.638008
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
68,21.060999,20.264281,20.662179,20.905340,21.723242,21.889035,21.532584,20.123357,21.212054,21.364028,...,9.285225,9.193118,10.508394,10.537868,10.560894,11.190901,12.029989,10.464184,10.176811,10.287341
69,14.339678,16.222470,16.403627,17.542362,17.307974,16.267054,15.547637,18.399281,18.879279,18.151313,...,5.954795,6.773327,13.500512,14.820114,13.452530,12.271348,10.644943,13.753358,12.053568,10.892689
70,17.946545,21.606356,20.881038,20.128038,20.142801,18.751223,15.977297,20.855200,21.301832,20.569131,...,17.241530,18.535290,21.058213,20.115118,18.468849,16.300276,16.706306,17.588500,16.473763,15.539892
71,16.691005,20.731803,19.212041,19.333126,18.879836,17.701594,15.846521,21.279786,19.640489,19.733636,...,13.494430,13.263348,15.935641,14.627894,13.250438,12.053713,12.388075,9.524722,9.797115,10.023031


In [6]:
# Standardizing dataframe
scaler = StandardScaler() 
variables = scaler.fit_transform(df) 
variables = pd.DataFrame(variables, columns = df.columns)
variables = variables*np.sqrt((len(variables)-1)/len(variables))
var_list = variables.columns

reset_columns = []
for _ in range(0,1012):
    reset_columns.append(str(_))
variables.columns = reset_columns

variables

,0,1,2,3,4,5,6,7,8,9,...,1002,1003,1004,1005,1006,1007,1008,1009,1010,1011
0,0.478755,0.064401,0.222762,0.291149,0.522996,0.525737,0.335813,-0.455956,-0.282817,-0.135470,...,-1.424142,-1.296521,-1.042626,-1.021682,-1.046136,-1.148263,-1.220699,-0.855694,-0.871314,-0.912676
1,-0.583490,-0.829198,-0.709578,-0.724076,-0.773658,-0.670103,-0.487952,-0.872072,-0.822637,-0.749074,...,-0.485956,-0.457092,-0.627805,-0.725776,-0.898526,-1.034818,-1.053715,-0.873443,-0.856178,-0.864434
2,-1.571588,-1.089926,-1.257178,-1.429033,-1.382471,-1.269193,-1.212619,-0.891614,-1.004984,-0.889931,...,-0.676331,-0.823302,-0.777875,-0.736736,-0.661490,-0.468105,-0.197749,-0.616922,-0.578995,-0.572429
3,-0.853225,-0.665235,-0.762639,-1.021185,-0.977769,-0.835819,-0.796134,-0.513788,-0.516976,-0.701561,...,2.308241,2.304139,-0.254320,-0.240623,-0.291846,-0.438260,-0.413276,-0.375283,-0.442989,-0.548526
4,-0.555796,-0.704184,-0.545022,-0.429491,-0.400924,-0.441644,-0.578900,-0.841656,-0.757211,-0.588262,...,-0.855491,-0.770613,-0.112563,0.287924,0.459806,0.553360,0.595601,-0.275829,-0.328319,-0.215958
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
68,0.636428,-0.324479,-0.126792,0.079585,0.358650,0.689609,1.101151,-0.396885,-0.289244,-0.096765,...,-1.106729,-1.117346,-1.013742,-1.065760,-1.063227,-0.919970,-0.712827,-1.048797,-1.149053,-1.171868
69,-1.055353,-1.179234,-1.057381,-0.714876,-0.731289,-0.789398,-0.731138,-0.733142,-0.735990,-0.772337,...,-1.862271,-1.637314,-0.321978,-0.059175,-0.314556,-0.605977,-1.127641,-0.288709,-0.659874,-0.999169
70,-0.147491,-0.040660,-0.078967,-0.104043,-0.031493,-0.135873,-0.599597,-0.254149,-0.272050,-0.263917,...,0.698238,0.890112,1.425328,1.185469,0.984215,0.564884,0.687708,0.597545,0.492256,0.326629
71,-0.463516,-0.225609,-0.443680,-0.291830,-0.343264,-0.412005,-0.639634,-0.171340,-0.590212,-0.439605,...,-0.151829,-0.242729,0.241013,-0.104358,-0.366880,-0.669225,-0.605582,-1.265896,-1.248021,-1.247273


In [7]:
correlations_matrix = variables.corr()
correlations_matrix

,0,1,2,3,4,5,6,7,8,9,...,1002,1003,1004,1005,1006,1007,1008,1009,1010,1011
0,1.000000,0.786168,0.826098,0.901964,0.962355,0.965407,0.932932,0.714918,0.758872,0.789322,...,0.122991,0.123580,0.038543,0.006292,0.043138,0.113606,0.154623,-0.015667,0.006591,0.033058
1,0.786168,1.000000,0.984900,0.939111,0.833038,0.748412,0.700746,0.964060,0.968729,0.955476,...,0.100341,0.102473,0.208821,0.188089,0.221353,0.257885,0.270094,0.162335,0.169798,0.173596
2,0.826098,0.984900,1.000000,0.972643,0.877978,0.791978,0.745731,0.934712,0.959720,0.971453,...,0.100104,0.101957,0.193744,0.170853,0.206865,0.249366,0.260827,0.144401,0.159070,0.167600
3,0.901964,0.939111,0.972643,1.000000,0.952540,0.881490,0.835230,0.878971,0.916883,0.951337,...,0.117523,0.126609,0.167205,0.138392,0.178353,0.232153,0.251247,0.110290,0.129797,0.143974
4,0.962355,0.833038,0.877978,0.952540,1.000000,0.977168,0.937739,0.769094,0.822225,0.867928,...,0.140361,0.144607,0.107635,0.074537,0.120245,0.191427,0.228020,0.049834,0.072635,0.093649
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1007,0.113606,0.257885,0.249366,0.232153,0.191427,0.183232,0.210665,0.263770,0.264128,0.276420,...,0.444099,0.427417,0.856832,0.877474,0.953504,1.000000,0.950730,0.850158,0.900120,0.924852
1008,0.154623,0.270094,0.260827,0.251247,0.228020,0.234596,0.271791,0.280141,0.278564,0.294380,...,0.442210,0.422632,0.699333,0.715423,0.829867,0.950730,1.000000,0.696748,0.780478,0.844009
1009,-0.015667,0.162335,0.144401,0.110290,0.049834,0.024997,0.041723,0.186443,0.181104,0.178121,...,0.361311,0.348619,0.925623,0.943313,0.937055,0.850158,0.696748,1.000000,0.982322,0.935349
1010,0.006591,0.169798,0.159070,0.129797,0.072635,0.050801,0.067981,0.188581,0.184958,0.190813,...,0.393269,0.373760,0.902764,0.918685,0.942347,0.900120,0.780478,0.982322,1.000000,0.983799


In [8]:
n = 1
corr_comb_f = []
while len(corr_comb_f) == 0:
    n += 1
    if n == 11:
        break

    ########################################
    z = []
    for a in range(0,1012):
        if len(correlations_matrix.iloc[:,a][correlations_matrix.iloc[:,a] >= 0.6]) >= 1012/n:
            z.append(str(a))

    ########################################
    i = 0

    while i < n:
        if i == 0:
            for _ in z:
                corr_comb_f.append([str(_)])
        else:
            corr_comb_i = corr_comb_f
            corr_comb_f = []
            for lista in corr_comb_i:
                a = int(lista[-1])
                for indx in correlations_matrix.iloc[a:,a][correlations_matrix.iloc[a:,a] < 0.6].index:
                    if indx in z:
                        corr_comb_f.append(lista + [indx])
        print(str(n)+"."+str(i+1))
        i += 1

2.1
2.2
3.1
3.2
3.3
4.1
4.2
4.3
4.4
5.1
5.2
5.3
5.4
5.5
6.1
6.2
6.3
6.4
6.5
6.6
7.1
7.2
7.3
7.4
7.5
7.6
7.7


In [9]:
print(len(corr_comb_f))
corr_comb_f

198418


[['214', '317', '474', '600', '615', '671', '814'],
 ['214', '317', '474', '600', '615', '671', '832'],
 ['214', '317', '474', '600', '615', '671', '833'],
 ['214', '317', '474', '600', '615', '671', '836'],
 ['214', '317', '474', '600', '615', '695', '814'],
 ['214', '317', '474', '600', '615', '695', '832'],
 ['214', '317', '474', '600', '615', '695', '833'],
 ['214', '317', '474', '600', '615', '695', '836'],
 ['214', '317', '474', '600', '615', '696', '832'],
 ['214', '317', '474', '600', '615', '696', '833'],
 ['214', '317', '474', '600', '615', '696', '836'],
 ['214', '317', '474', '600', '615', '697', '715'],
 ['214', '317', '474', '600', '615', '697', '716'],
 ['214', '317', '474', '600', '615', '697', '832'],
 ['214', '317', '474', '600', '615', '697', '833'],
 ['214', '317', '474', '600', '615', '697', '836'],
 ['214', '317', '474', '600', '615', '721', '836'],
 ['214', '317', '474', '600', '638', '645', '813'],
 ['214', '317', '474', '600', '638', '645', '814'],
 ['214', '31

In [10]:
z

['214',
 '215',
 '216',
 '235',
 '236',
 '237',
 '238',
 '253',
 '254',
 '255',
 '256',
 '257',
 '258',
 '272',
 '273',
 '274',
 '275',
 '276',
 '290',
 '291',
 '292',
 '293',
 '294',
 '295',
 '296',
 '297',
 '309',
 '310',
 '311',
 '312',
 '313',
 '314',
 '315',
 '316',
 '317',
 '329',
 '330',
 '331',
 '332',
 '333',
 '334',
 '335',
 '336',
 '349',
 '350',
 '351',
 '352',
 '353',
 '354',
 '355',
 '356',
 '357',
 '358',
 '369',
 '370',
 '371',
 '372',
 '373',
 '374',
 '375',
 '376',
 '377',
 '390',
 '391',
 '392',
 '393',
 '394',
 '395',
 '396',
 '397',
 '410',
 '411',
 '412',
 '413',
 '414',
 '415',
 '416',
 '417',
 '418',
 '429',
 '430',
 '431',
 '432',
 '433',
 '434',
 '435',
 '436',
 '437',
 '450',
 '451',
 '452',
 '453',
 '454',
 '455',
 '456',
 '457',
 '458',
 '470',
 '471',
 '472',
 '473',
 '474',
 '475',
 '476',
 '477',
 '478',
 '491',
 '492',
 '493',
 '494',
 '495',
 '496',
 '497',
 '498',
 '499',
 '512',
 '513',
 '514',
 '515',
 '516',
 '517',
 '518',
 '519',
 '533',
 '534',
